In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#Here is the recreated **Kaggle Notebook 17** (`17_otel_graphistry_trace_glue_e1.py`). This script merges the real inference and observability pipeline from Notebook 14 with the strict correlation and artifact generation logic of Notebook 17.

# -*- coding: utf-8 -*-
"""17-otel-graphistry-trace-glue-e1.ipynb

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1FZ-gCb58Yi52I4nY6AX1rKgwfs9gDIyY

# 17 - OTEL + Graphistry Trace Glue (Production Implementation)

This notebook creates a **stable join key** between OTEL traces (Grafana) and
Graphistry trace graphs by introducing `run_id` and `trace_id`, then persists a
small artifact (`manifest.json` + `spans.parquet`) that links the two worlds.

It merges:
- **Notebook 14**: Real GGUF Inference, Dual-GPU logic, and OpenTelemetry pipeline.
- **Notebook 17 Logic**: Run ID generation, SHA256 correlation, and Artifact export.
"""

In [1]:
# ==============================================================================
# Step 1: Verify Dual GPU Environment
# ==============================================================================
import subprocess
import os

print("="*70)
print("🔍 SPLIT-GPU ENVIRONMENT CHECK")
print("="*70)

result = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free", "--format=csv,noheader"],
    capture_output=True, text=True
)

gpus = result.stdout.strip().split('\n')
print(f"\n📊 Detected {len(gpus)} GPU(s):")
for gpu in gpus:
    print(f"   {gpu}")

if len(gpus) >= 2:
    print("\n✅ Dual T4 ready for split-GPU operation!")
    print("   GPU 0 → llama-server (GGUF model inference)")
    print("   GPU 1 → RAPIDS/Graphistry (architecture visualization)")
else:
    print("\n⚠️ Need 2 GPUs for split operation")

🔍 SPLIT-GPU ENVIRONMENT CHECK

📊 Detected 2 GPU(s):
   0, Tesla T4, 15360 MiB, 14913 MiB
   1, Tesla T4, 15360 MiB, 14913 MiB

✅ Dual T4 ready for split-GPU operation!
   GPU 0 → llama-server (GGUF model inference)
   GPU 1 → RAPIDS/Graphistry (architecture visualization)


In [3]:
# ==============================================================================
# Step 2: Install Dependencies
# ==============================================================================
print("📦 Installing dependencies...")

# Install llamatelemetry v1.0.0
!pip install -q https://github.com/llamatelemetry/llamatelemetry/releases/download/v1.0.0/llamatelemetry-v1.0.0-source.tar.gz

# Install RAPIDS (GPU 1) and Graphistry
!pip install -q --extra-index-url=https://pypi.nvidia.com "cugraph-cu12==25.6.*" "cudf-cu12==25.6.*"
!pip install -q "graphistry[ai]"

# Install OTel and Utilities
!pip install -q pandas numpy scipy huggingface_hub
!pip install -q \
opentelemetry-api==1.37.0 \
opentelemetry-sdk==1.37.0 \
opentelemetry-proto==1.37.0 \
opentelemetry-exporter-otlp-proto-common==1.37.0 \
opentelemetry-exporter-otlp-proto-grpc==1.37.0 \
rich \
--upgrade-strategy=only-if-needed

# Verify installations
import llamatelemetry
import cudf, cugraph
import graphistry
print(f"✅ llamatelemetry {llamatelemetry.__version__}")
print(f"✅ cuDF {cudf.__version__}")
print(f"✅ Graphistry {graphistry.__version__}")

📦 Installing dependencies...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 67.5 MB/s eta 0:00:00:00:010:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 43.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 MB 14.5 MB/s eta 0:00:0000:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
datasets 4.4.2 requires pyarrow>=21.0.0, but you have pyarrow 19.0.1 which is incompatible.
bigframes 2.26.0 requires rich<14,>=12.4.4, but you have rich 14.2.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:979: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


v1.0.0/llamatelemetry-v1.0.0-cuda12-kagg(…):   0%|          | 0.00/1.40G [00:00<?, ?B/s]

📦 Extracting llamatelemetry-v1.0.0-cuda12-kaggle-t4x2.tar.gz...
Found 19 files in archive
Extracted 19 files to /root/.cache/llamatelemetry/extract_1.0.0
✅ Extraction complete!
  Found bin/ and lib/ under /root/.cache/llamatelemetry/extract_1.0.0/llamatelemetry-v1.0.0-cuda12-kaggle-t4x2
  Copied 13 binaries to /usr/local/lib/python3.12/dist-packages/llamatelemetry/binaries/cuda12
  Copied 2 libraries to /usr/local/lib/python3.12/dist-packages/llamatelemetry/lib
✅ Binaries installed successfully!

✅ llamatelemetry 1.0.0
✅ cuDF 25.06.00
✅ Graphistry 0.50.6


In [18]:
import numpy as np
import pandas as pd

In [20]:
# ==============================================================================
# Step 3: Setup Secrets & Run Context (The Glue) - COMPLETE REWRITE
# ==============================================================================
import os
import json
import uuid
import hashlib
import platform
import base64
import requests  # Important: Add requests for connection testing
from datetime import datetime, timezone
from kaggle_secrets import UserSecretsClient

print("="*70)
print("🔐 STEP 3: SETUP SECRETS & RUN CONTEXT")
print("="*70)

secrets = UserSecretsClient()

# --- Glue Logic: Stable Run Identity ---
RUN_ID = os.getenv("LLAMATELEMETRY_RUN_ID") or f"run_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}_{uuid.uuid4().hex[:8]}"
os.environ["LLAMATELEMETRY_RUN_ID"] = RUN_ID

def _h(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()[:16]

HOST_ID = _h(platform.node() + "|" + platform.platform())

print(f"\n🔗 RUN_ID: {RUN_ID}")
print(f"🖥️ HOST_ID: {HOST_ID}")

# ==============================================================================
# Grafana OTLP Configuration
# ==============================================================================
print("\n📡 GRAFANA OTLP CONFIGURATION:")

try:
    GRAFANA_OTLP_ENDPOINT = secrets.get_secret("GRAFANA_OTLP_ENDPOINT").rstrip("/")
    GRAFANA_OTLP_BASIC_B64 = secrets.get_secret("GRAFANA_OTLP_BASIC_B64")
    
    print(f"   Endpoint: {GRAFANA_OTLP_ENDPOINT}")
    print(f"   Auth Token: {GRAFANA_OTLP_BASIC_B64[:15]}... (length: {len(GRAFANA_OTLP_BASIC_B64)})")
    
    # Verify the Basic Auth format (should be base64 of "username:password")
    try:
        decoded = base64.b64decode(GRAFANA_OTLP_BASIC_B64).decode('utf-8')
        if ':' in decoded:
            username = decoded.split(':')[0]
            print(f"   ✅ Auth format valid (username: {username})")
        else:
            print(f"   ⚠️ Auth missing colon separator - should be 'username:password' base64 encoded")
    except:
        print(f"   ⚠️ Auth is not valid base64 - please check your secret")
        
except Exception as e:
    print(f"   ❌ Failed to retrieve Grafana secrets: {e}")
    GRAFANA_OTLP_ENDPOINT = None
    GRAFANA_OTLP_BASIC_B64 = None

# ==============================================================================
# Test Grafana Connection (Proper OTLP Test)
# ==============================================================================
print("\n🔍 TESTING GRAFANA CONNECTION:")

if GRAFANA_OTLP_ENDPOINT and GRAFANA_OTLP_BASIC_B64:
    
    # Test 1: Basic connectivity with OPTIONS request
    try:
        options_response = requests.options(
            f"{GRAFANA_OTLP_ENDPOINT}/v1/traces",
            headers={"Authorization": f"Basic {GRAFANA_OTLP_BASIC_B64}"},
            timeout=5
        )
        print(f"   OPTIONS test - Status: {options_response.status_code}")
        
        if options_response.status_code == 405:
            print("   ✅ Endpoint exists (405 is expected for OPTIONS)")
        elif options_response.status_code == 200:
            print("   ✅ Endpoint fully accessible")
        elif options_response.status_code == 401:
            print("   ❌ Authentication failed - check your credentials")
        elif options_response.status_code == 404:
            print("   ❌ Endpoint not found - check your URL")
            
    except requests.exceptions.ConnectionError:
        print("   ❌ Cannot connect to endpoint - check URL or network")
    except requests.exceptions.Timeout:
        print("   ❌ Connection timeout - endpoint may be slow or unreachable")
    except Exception as e:
        print(f"   ❌ Connection test failed: {e}")
    
    # Test 2: Send a minimal valid OTLP payload (JSON format for testing)
    print("\n   Sending test OTLP payload...")
    
    # Minimal valid OTLP JSON payload
    test_payload = {
        "resourceSpans": [{
            "resource": {
                "attributes": [
                    {"key": "service.name", "value": {"stringValue": "connection-test"}},
                    {"key": "test.run_id", "value": {"stringValue": RUN_ID}}
                ]
            },
            "scopeSpans": [{
                "scope": {"name": "test-scope"},
                "spans": []
            }]
        }]
    }
    
    try:
        post_response = requests.post(
            f"{GRAFANA_OTLP_ENDPOINT}/v1/traces",
            headers={
                "Authorization": f"Basic {GRAFANA_OTLP_BASIC_B64}",
                "Content-Type": "application/json"
            },
            json=test_payload,
            timeout=10
        )
        
        print(f"   POST test - Status: {post_response.status_code}")
        
        if post_response.status_code == 200:
            print("   ✅ OTLP endpoint is working! (JSON test successful)")
        elif post_response.status_code == 400:
            print("   ⚠️ Got 400 - This is normal if endpoint expects protobuf")
            print("      The actual exporter will use protobuf format")
            print("      ✅ Connection and auth are working!")
        elif post_response.status_code == 415:
            print("   ⚠️ Got 415 - Unsupported media type (expects protobuf)")
            print("      ✅ Connection and auth are working!")
        else:
            print(f"   ⚠️ Unexpected response: {post_response.status_code}")
            if hasattr(post_response, 'text'):
                print(f"   Response: {post_response.text[:200]}")
                
    except Exception as e:
        print(f"   ❌ POST test failed: {e}")
        
else:
    print("   ⚠️ Skipping connection test - Grafana secrets not available")

# ==============================================================================
# HuggingFace Configuration
# ==============================================================================
print("\n🤗 HUGGINGFACE CONFIGURATION:")

try:
    HF_TOKEN = secrets.get_secret("HF_TOKEN")
    print(f"   Token: {HF_TOKEN[:10]}... (length: {len(HF_TOKEN)})")
    
    # Test HF token (optional but recommended)
    from huggingface_hub import HfApi
    api = HfApi()
    user_info = api.whoami(token=HF_TOKEN)
    print(f"   ✅ Logged in as: {user_info.get('name', user_info.get('login', 'unknown'))}")
    
except Exception as e:
    print(f"   ❌ Failed to retrieve HF_TOKEN: {e}")
    HF_TOKEN = None

# ==============================================================================
# Graphistry Configuration
# ==============================================================================
print("\n📊 GRAPHISTRY CONFIGURATION:")

try:
    GRAPHISTRY_PERSONAL_KEY_ID = secrets.get_secret("Graphistry_Personal_Key_ID")
    GRAPHISTRY_PERSONAL_KEY_SECRET = secrets.get_secret("Graphistry_Personal_Secret_Key")
    
    print(f"   Key ID: {GRAPHISTRY_PERSONAL_KEY_ID[:10]}...")
    print(f"   Key Secret: {GRAPHISTRY_PERSONAL_KEY_SECRET[:5]}... (length: {len(GRAPHISTRY_PERSONAL_KEY_SECRET)})")
    
except Exception as e:
    print(f"   ❌ Failed to retrieve Graphistry secrets: {e}")
    GRAPHISTRY_PERSONAL_KEY_ID = None
    GRAPHISTRY_PERSONAL_KEY_SECRET = None

# ==============================================================================
# Login/Register Services
# ==============================================================================
print("\n🔐 LOGGING INTO SERVICES:")

# HuggingFace Login
if HF_TOKEN:
    try:
        from huggingface_hub import login
        login(HF_TOKEN, add_to_git_credential=False)
        print("   ✅ HuggingFace login successful")
    except Exception as e:
        print(f"   ❌ HuggingFace login failed: {e}")
else:
    print("   ⚠️ Skipping HuggingFace login - no token")

# Graphistry Registration
if GRAPHISTRY_PERSONAL_KEY_ID and GRAPHISTRY_PERSONAL_KEY_SECRET:
    try:
        import graphistry
        graphistry.register(
            api=3,
            protocol="https",
            server="hub.graphistry.com",
            personal_key_id=GRAPHISTRY_PERSONAL_KEY_ID,
            personal_key_secret=GRAPHISTRY_PERSONAL_KEY_SECRET,
        )
        print("   ✅ Graphistry registration successful")
        
        # Verify Graphistry connection
        test_url = graphistry.edges(pd.DataFrame({'src': ['a'], 'dst': ['b']})).plot(render=False)
        print(f"   ✅ Graphistry test URL generated")
        
    except Exception as e:
        print(f"   ❌ Graphistry registration failed: {e}")
else:
    print("   ⚠️ Skipping Graphistry registration - missing credentials")

# ==============================================================================
# Summary
# ==============================================================================
print("\n" + "="*70)
print("📋 STEP 3 SUMMARY")
print("="*70)
print(f"RUN_ID: {RUN_ID}")
print(f"HOST_ID: {HOST_ID}")
print(f"Grafana Endpoint: {GRAFANA_OTLP_ENDPOINT or '❌ Missing'}")
print(f"Grafana Auth: {'✅ Present' if GRAFANA_OTLP_BASIC_B64 else '❌ Missing'}")
print(f"HF Token: {'✅ Present' if HF_TOKEN else '❌ Missing'}")
print(f"Graphistry: {'✅ Configured' if GRAPHISTRY_PERSONAL_KEY_ID else '❌ Missing'}")
print("="*70)

# Export critical variables for next steps
# (These are already in the Python environment)

🔐 STEP 3: SETUP SECRETS & RUN CONTEXT

🔗 RUN_ID: run_20260217T073415Z_e5c3e808
🖥️ HOST_ID: af18db94d43dcec6

📡 GRAFANA OTLP CONFIGURATION:
   Endpoint: https://otlp-gateway-prod-us-east-0.grafana.net/otlp
   Auth Token: ODU4OTA3OmdsY19... (length: 220)
   ✅ Auth format valid (username: 858907)

🔍 TESTING GRAFANA CONNECTION:
   OPTIONS test - Status: 405
   ✅ Endpoint exists (405 is expected for OPTIONS)

   Sending test OTLP payload...
   POST test - Status: 200
   ✅ OTLP endpoint is working! (JSON test successful)

🤗 HUGGINGFACE CONFIGURATION:
   Token: hf_BjFWKGr... (length: 37)
   ✅ Logged in as: waqasm86

📊 GRAPHISTRY CONFIGURATION:
   Key ID: 0B5H7XL4SS...
   Key Secret: BU8Q4... (length: 16)

🔐 LOGGING INTO SERVICES:
   ✅ HuggingFace login successful
   ✅ Graphistry registration successful
   ❌ Graphistry registration failed: Both "source" and "destination" must be bound before plotting.

📋 STEP 3 SUMMARY
RUN_ID: run_20260217T073415Z_e5c3e808
HOST_ID: af18db94d43dcec6
Grafana End

In [5]:
# ==============================================================================
# Step 4: Model Download & SHA256 Calculation (Correlation Key)
# ==============================================================================
from huggingface_hub import hf_hub_download
from pathlib import Path

# Create models directory
os.makedirs("/kaggle/working/models", exist_ok=True)

# Download model
print("Downloading model...")
model_path = hf_hub_download(
    repo_id="bartowski/Qwen2.5-3B-Instruct-GGUF",
    filename="Qwen2.5-3B-Instruct-Q4_K_M.gguf",
    local_dir="/kaggle/working/models",
)
print(f"✓ Model downloaded: {model_path}")

# --- Glue Logic: Model Integrity Check ---
def file_sha256(path: str) -> str:
    p = Path(path)
    if not p.exists() or not p.is_file():
        return ""
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

GGUF_SHA = file_sha256(model_path)
print(f"🔑 GGUF SHA256: {GGUF_SHA[:16]}...")

# Define Runtime Metadata (Used in every span)
RUNTIME_META = {
    "run_id": RUN_ID,
    "host_id": HOST_ID,
    "model.name": "Qwen2.5-3B-Instruct",
    "model.quant": "Q4_K_M",
    "model.gguf_sha256": GGUF_SHA,
    "gpu.tensor_split": "1.0",
    "environment": "kaggle-notebook"
}

Qwen2.5-3B-Instruct-Q4_K_M.gguf:   0%|          | 0.00/1.93G [00:00<?, ?B/s]

✓ Model downloaded: /kaggle/working/models/Qwen2.5-3B-Instruct-Q4_K_M.gguf
🔑 GGUF SHA256: 9c9f56a391a3abbd...


In [21]:
# ==============================================================================
# Step 5: OpenTelemetry Setup (FIXED with better error handling and debug)
# ==============================================================================
import logging
from opentelemetry import trace, metrics
from opentelemetry.sdk.resources import Resource
from opentelemetry.sdk.trace import TracerProvider
from opentelemetry.sdk.trace.export import BatchSpanProcessor, SimpleSpanProcessor
from opentelemetry.exporter.otlp.proto.http.trace_exporter import OTLPSpanExporter
from opentelemetry.sdk.trace.export.in_memory_span_exporter import InMemorySpanExporter
from opentelemetry.sdk.metrics import MeterProvider
from opentelemetry.sdk.metrics.export import PeriodicExportingMetricReader
from opentelemetry.exporter.otlp.proto.http.metric_exporter import OTLPMetricExporter
from opentelemetry.sdk.trace.export import ConsoleSpanExporter  # 🔴 NEW: For debugging

# Silence OTel logs
logging.getLogger("opentelemetry").setLevel(logging.WARNING)

# Create Resource with Glue Metadata
resource = Resource.create({
    "service.name": "llamatelemetry-inference",
    "service.version": "1.0.0",
    **RUNTIME_META
})

# Tracer Provider
tracer_provider = TracerProvider(resource=resource)

# 🔴 NEW: Console exporter for debugging (REMOVE after fixing)
console_exporter = ConsoleSpanExporter()
tracer_provider.add_span_processor(SimpleSpanProcessor(console_exporter))
print("   ✅ Console exporter added for debugging")

# 1. Grafana Exporter - with detailed error handling
print("\n📡 Setting up Grafana OTLP exporter...")
try:
    span_exporter_grafana = OTLPSpanExporter(
        endpoint=f"{GRAFANA_OTLP_ENDPOINT}/v1/traces",
        headers={
            "Authorization": f"Basic {GRAFANA_OTLP_BASIC_B64}",
            "Content-Type": "application/x-protobuf",
        },
        timeout=10
    )
    # Test the exporter with a simple export
    tracer_provider.add_span_processor(BatchSpanProcessor(span_exporter_grafana))
    print("   ✅ Grafana trace exporter configured")
except Exception as e:
    print(f"   ❌ Failed to configure Grafana exporter: {e}")
    print("   Will continue with in-memory and console exporters only")

# 2. In-Memory Exporter (for Graphistry)
memory_exporter = InMemorySpanExporter()
tracer_provider.add_span_processor(SimpleSpanProcessor(memory_exporter))
print("   ✅ In-memory exporter configured")

trace.set_tracer_provider(tracer_provider)
tracer = trace.get_tracer(__name__)

# Metrics Setup with better error handling
try:
    metric_exporter = OTLPMetricExporter(
        endpoint=f"{GRAFANA_OTLP_ENDPOINT}/v1/metrics",
        headers={"Authorization": f"Basic {GRAFANA_OTLP_BASIC_B64}"},
        timeout=10
    )
    meter_provider = MeterProvider(
        resource=resource,
        metric_readers=[PeriodicExportingMetricReader(metric_exporter, export_interval_millis=5000)]
    )
    metrics.set_meter_provider(meter_provider)
    meter = metrics.get_meter(__name__)
    print("   ✅ Metrics configured")
    
    # Custom Instruments
    request_counter = meter.create_counter("llm.requests.total", unit="1", description="Total LLM requests")
    latency_histogram = meter.create_histogram("llm.request.duration", unit="ms", description="Request latency")
except Exception as e:
    print(f"   ⚠️ Metrics setup failed: {e}")
    # Fallback to no-op meter
    meter = metrics.get_meter(__name__)
    request_counter = None
    latency_histogram = None

Overriding of current TracerProvider is not allowed
Overriding of current MeterProvider is not allowed


   ✅ Console exporter added for debugging

📡 Setting up Grafana OTLP exporter...
   ✅ Grafana trace exporter configured
   ✅ In-memory exporter configured
   ✅ Metrics configured


In [7]:
# ==============================================================================
# Step 6: Dual GPU Inference Setup (GPU 0)
# ==============================================================================
import torch
from llamatelemetry.server import ServerManager

# Check GPUs
print(f"\nFound {torch.cuda.device_count()} GPUs:")

# Start server on GPU 0
server = ServerManager(server_url="http://127.0.0.1:8090")
server.start_server(
    model_path=model_path,
    gpu_layers=99,
    tensor_split="1.0",  # Use only first GPU
    port=8090,
    host="127.0.0.1",
    ctx_size=4096,
    batch_size=512,
)
print("\n✓ Server running on http://127.0.0.1:8090")
print("✓ GPU 0: Used for LLM")
print("✓ GPU 1: Free for Graphistry")


Found 2 GPUs:
GPU Check:
  Platform: kaggle
  GPU: Tesla T4
  Compute Capability: 7.5
  Status: ✓ Compatible
Starting llama-server...
  Executable: /usr/local/lib/python3.12/dist-packages/llamatelemetry/binaries/cuda12/llama-server
  Model: Qwen2.5-3B-Instruct-Q4_K_M.gguf
  GPU Layers: 99
  Context Size: 4096
  Server URL: http://127.0.0.1:8090
Waiting for server to be ready...... ✓ Ready in 3.0s

✓ Server running on http://127.0.0.1:8090
✓ GPU 0: Used for LLM
✓ GPU 1: Free for Graphistry


In [22]:
# ==============================================================================
# Step 7: Instrumented Inference with Glue Attributes
# ==============================================================================
from llamatelemetry.api import LlamaCppClient
from opentelemetry.trace import Status, StatusCode
import time

class InstrumentedLLMClient:
    """LLM client with OpenTelemetry instrumentation + Glue Metadata"""
    def __init__(self, base_url: str, tracer, meter):
        self.client = LlamaCppClient(base_url)
        self.tracer = tracer
        # FIX: Create the meter attributes in __init__
        self.request_counter = meter.create_counter("llm.requests.total", unit="1")
        self.latency_histogram = meter.create_histogram("llm.request.duration", unit="ms")

    def chat_completion(self, messages: list, **kwargs):
        model = kwargs.get("model", "unknown")
        
        with self.tracer.start_as_current_span(
            name=f"llm.request.{model}",
            kind=trace.SpanKind.CLIENT,
            attributes=RUNTIME_META # Inject Glue Metadata
        ) as span:
            try:
                span.set_attribute("llm.request.messages", len(messages))
                
                start_time = time.time()
                response = self.client.chat.completions.create(messages=messages, **kwargs)
                latency_ms = (time.time() - start_time) * 1000
                
                content = response.choices[0].message.content
                
                # Metrics
                span.set_attribute("llm.response.length", len(content))
                self.request_counter.add(1, attributes={"model": model, "status": "success"})
                self.latency_histogram.record(latency_ms, attributes={"model": model})
                
                span.set_status(Status(StatusCode.OK))
                return response

            except Exception as e:
                span.set_status(Status(StatusCode.ERROR, str(e)))
                span.record_exception(e)
                raise

# Initialize client
llm = InstrumentedLLMClient("http://127.0.0.1:8090", tracer, meter)

In [12]:
# ==============================================================================
# Step 8: Execute Batch Requests
# ==============================================================================
import random

# Wrap in parent span
with tracer.start_as_current_span("llm.batch.requests", attributes=RUNTIME_META) as batch_span:
    
    # 1. Initial request
    response = llm.chat_completion(
        messages=[{"role": "user", "content": "What is CUDA?"}],
        max_tokens=100,
    )
    print(f"Response: {response.choices[0].message.content[:50]}...")

    # 2. Batch requests
    prompts = [
        "Explain transformer architecture",
        "What is quantization in LLMs?",
        "How does FlashAttention work?",
        "Describe the attention mechanism",
        "What is GGUF format?",
    ]

    responses = []
    for i, prompt in enumerate(prompts):
        print(f"Request {i+1}/{len(prompts)}: {prompt[:30]}...")
        resp = llm.chat_completion(
            messages=[{"role": "user", "content": prompt}],
            max_tokens=random.randint(50, 150),
        )
        responses.append(resp)
        time.sleep(0.2)

    print(f"Completed {len(responses)} requests")

# Force flush to ensure spans are captured
tracer_provider.force_flush()

Response: CUDA (Compute Unified Device Architecture) is a pa...
Request 1/5: Explain transformer architectu...
Request 2/5: What is quantization in LLMs?...
Request 3/5: How does FlashAttention work?...
Request 4/5: Describe the attention mechani...
Request 5/5: What is GGUF format?...
Completed 5 requests


True

In [23]:
# ==============================================================================
# Step 8b: Execute Batch Requests (with post-verification)
# ==============================================================================

# ... (your existing batch request code) ...

# Force flush to ensure spans are captured
tracer_provider.force_flush()
print("🔄 Flushed spans to exporters")

# 🔴 NEW: Verify spans were captured
from opentelemetry import trace
import time

# Give exporters a moment to send
time.sleep(2)

# Check memory exporter
finished_spans = memory_exporter.get_finished_spans()
print(f"\n📊 Captured {len(finished_spans)} spans in memory")

if len(finished_spans) > 0:
    print("\n📋 First 3 spans captured:")
    for i, span in enumerate(finished_spans[:3]):
        print(f"   {i+1}. {span.name} (trace: {format(span.context.trace_id, '032x')[:8]}...)")
        
    # Check if any have the run_id attribute
    spans_with_run_id = [s for s in finished_spans if s.attributes and s.attributes.get("run_id") == RUN_ID]
    print(f"\n   ✓ {len(spans_with_run_id)} spans have correct run_id: {RUN_ID}")
else:
    print("\n❌ No spans captured in memory - check instrumentation")

🔄 Flushed spans to exporters

📊 Captured 0 spans in memory

❌ No spans captured in memory - check instrumentation


In [13]:
# ==============================================================================
# Step 9: Transform Spans to Graph Data (GPU 1)
# ==============================================================================
import rmm
# Switch RAPIDS to GPU 1
rmm.reinitialize(devices=[1])
import cudf

finished_spans = memory_exporter.get_finished_spans()
print(f"Captured {len(finished_spans)} spans")

span_data = []
for span in finished_spans:
    # Extract Glue Keys
    attrs = dict(span.attributes) if span.attributes else {}
    
    span_data.append({
        "span_id": format(span.context.span_id, "016x"),
        "parent_span_id": format(span.parent.span_id, "016x") if span.parent else None,
        "trace_id": format(span.context.trace_id, "032x"),
        "name": span.name,
        "start_time": span.start_time,
        "end_time": span.end_time,
        "duration_ms": (span.end_time - span.start_time) / 1_000_000,
        "status": span.status.status_code.name,
        # Glue Attributes
        "run_id": attrs.get("run_id", "unknown"),
        "gguf_sha": attrs.get("model.gguf_sha256", "unknown"),
    })

df_spans = cudf.DataFrame(span_data)
print(f"Span DataFrame shape: {df_spans.shape}")

# Create Edges
if len(df_spans) > 0:
    df_edges = df_spans[df_spans["parent_span_id"].notnull()][
        ["parent_span_id", "span_id", "trace_id"]
    ].rename(columns={
        "parent_span_id": "source",
        "span_id": "destination",
    })
else:
    df_edges = cudf.DataFrame(columns=["source", "destination", "trace_id"])

print(f"Edges DataFrame shape: {df_edges.shape}")

Captured 9 spans
Span DataFrame shape: (9, 10)
Edges DataFrame shape: (7, 3)


In [14]:
# ==============================================================================
# Step 10: Graphistry Trace Graph Visualization
# ==============================================================================
if len(df_edges) > 0:
    g = graphistry.edges(df_edges, "source", "destination")
    g = g.nodes(df_spans, "span_id")
    
    # Bind visual encodings
    g = g.bind(
        point_title="name",
        point_size="duration_ms",
        point_color="status",
        edge_title="trace_id"
    )
    
    # Color encoding for status
    g = g.encode_point_color("status", categorical_mapping={
        "OK": "#4CAF50",      # Green
        "ERROR": "#F44336",   # Red
        "UNSET": "#9E9E9E"    # Grey
    }, as_categorical=True)
    
    # Generate URL
    url = g.plot(render=False)
    print(f"🔗 Trace Graph Dashboard: {url}")
else:
    print("No edges available for graph visualization")
    url = "no_graph_generated"

🔗 Trace Graph Dashboard: https://hub.graphistry.com/graph/graph.html?dataset=c1e89702e5fc4bb0b354cf8fdbdf418a&type=arrow&viztoken=0fac279f-5758-4bf2-adeb-9a52a7159a51&usertag=2ea92f7a-pygraphistry-0.50.6&splashAfter=1771314015&info=true


In [15]:
# ==============================================================================
# Step 11: Persist "Trace-to-Graph" Artifact (The Glue Export)
# ==============================================================================
from pathlib import Path

OUT_DIR = Path("/kaggle/working/llamatelemetry_artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Select one trace_id for the manifest example (usually the root or batch)
primary_trace_id = df_spans["trace_id"].iloc[0] if len(df_spans) > 0 else "unknown"

artifact = {
    "run_id": RUN_ID,
    "trace_id": primary_trace_id,
    "runtime_meta": RUNTIME_META,
    "graphistry_trace_url": str(url),
    "created_utc": datetime.now(timezone.utc).isoformat(),
    "span_count": len(df_spans),
    "edge_count": len(df_edges)
}

# 1. Write Manifest
manifest_path = OUT_DIR / f"{RUN_ID}_{primary_trace_id}_manifest.json"
manifest_path.write_text(json.dumps(artifact, indent=2))

# 2. Write Spans Parquet (for offline analysis)
parquet_path = OUT_DIR / f"{RUN_ID}_{primary_trace_id}_spans.parquet"
# Convert cudf to pandas for parquet export if needed (cudf supports parquet directly)
df_spans.to_parquet(parquet_path)

print("\n" + "="*70)
print("📦 ARTIFACT EXPORT COMPLETE")
print("="*70)
print(f"Manifest: {manifest_path}")
print(f"Spans: {parquet_path}")
print("\nManifest Contents:")
print(json.dumps(artifact, indent=2))


📦 ARTIFACT EXPORT COMPLETE
Manifest: /kaggle/working/llamatelemetry_artifacts/run_20260217T073415Z_e5c3e808_03de258a127c1d2df634ba3673746988_manifest.json
Spans: /kaggle/working/llamatelemetry_artifacts/run_20260217T073415Z_e5c3e808_03de258a127c1d2df634ba3673746988_spans.parquet

Manifest Contents:
{
  "run_id": "run_20260217T073415Z_e5c3e808",
  "trace_id": "03de258a127c1d2df634ba3673746988",
  "runtime_meta": {
    "run_id": "run_20260217T073415Z_e5c3e808",
    "host_id": "af18db94d43dcec6",
    "model.name": "Qwen2.5-3B-Instruct",
    "model.quant": "Q4_K_M",
    "model.gguf_sha256": "9c9f56a391a3abbd5b89d0245bf6106081bcc3173119d4229235dd9d23253f94",
    "gpu.tensor_split": "1.0",
    "environment": "kaggle-notebook"
  },
  "graphistry_trace_url": "https://hub.graphistry.com/graph/graph.html?dataset=c1e89702e5fc4bb0b354cf8fdbdf418a&type=arrow&viztoken=0fac279f-5758-4bf2-adeb-9a52a7159a51&usertag=2ea92f7a-pygraphistry-0.50.6&splashAfter=1771314015&info=true",
  "created_utc": "2026-